In [1]:
import duckdb

con = duckdb.connect()

In [4]:
books_path = ("C:/Users/Quang/Documents/USTH/FundDS/dataset/goodreads_books.json.gz")
authors_path = ("C:/Users/Quang/Documents/USTH/FundDS/dataset/goodreads_book_authors.json.gz")
works_path = ("C:/Users/Quang/Documents/USTH/FundDS/dataset/goodreads_book_works.json.gz")
genres_path = ("C:/Users/Quang/Documents/USTH/FundDS/dataset/goodreads_book_genres_initial.json.gz")
series_path = ("C:/Users/Quang/Documents/USTH/FundDS/dataset/goodreads_book_series.json.gz")
interactions_path = ("C:/Users/Quang/Documents/USTH/FundDS/dataset/goodreads_interactions.parquet")

In [16]:
con.sql("""--sql
    SELECT 
        TRY_CAST(book_id AS INTEGER) AS book_id,
        title AS book_title,
        title_without_series AS book_title_without_series,
        country_code AS book_country_code,
        language_code AS book_language_code,
        TRY_CAST(average_rating AS FLOAT) AS book_average_rating,
        TRY_CAST(ratings_count AS INTEGER) AS book_ratings_count,
        format AS book_format,
        publisher AS book_publisher,
        TRY_CAST(num_pages AS INTEGER) AS book_num_pages,
        TRY_CAST(publication_year AS INTEGER) AS book_publication_year,
        url AS book_url ,
        image_url AS book_image_url ,
        TRY_CAST(work_id AS INTEGER) AS work_id ,
    FROM read_json($1)
    LIMIT 5
    """,
    params=[books_path],
).show()

┌─────────┬──────────────────────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────────────────┬───────────────────┬────────────────────┬─────────────────────┬────────────────────┬─────────────┬────────────────────────┬────────────────┬───────────────────────┬───────────────────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────┬─────────┐
│ book_id │                              book_title                              │                      book_title_without_series                       │ book_country_code │ book_language_code │ book_average_rating │ book_ratings_count │ book_format │     book_publisher     │ book_num_pages │ book_publication_year │                             book_url                              │                                      book_image_url                                      │ work_id │
│  int32  │               

In [18]:
con.sql("""--sql
    SELECT 
        TRY_CAST(work_id AS INTEGER) AS work_id,
        original_title AS work_title,
        TRY_CAST(books_count AS INTEGER) AS work_books_count,
        TRY_CAST(original_publication_year AS INTEGER) AS book_publication_year,
        TRY_CAST(best_book_id AS INTEGER) AS work_best_book_id,
        TRY_CAST(ratings_count AS INTEGER) AS work_ratings_count,
        TRY_CAST(ratings_sum AS INTEGER) AS work_ratings_sum,
    FROM read_json($1)
    LIMIT 5
    """,
    params=[works_path],
).show()

┌─────────┬──────────────────────────────────────────────────────────────────────┬──────────────────┬───────────────────────┬───────────────────┬────────────────────┬──────────────────┐
│ work_id │                              work_title                              │ work_books_count │ book_publication_year │ work_best_book_id │ work_ratings_count │ work_ratings_sum │
│  int32  │                               varchar                                │      int32       │         int32         │       int32       │       int32        │      int32       │
├─────────┼──────────────────────────────────────────────────────────────────────┼──────────────────┼───────────────────────┼───────────────────┼────────────────────┼──────────────────┤
│ 5400751 │ W. C. Fields: A Life on Film                                         │                1 │                  1984 │           5333265 │                  3 │               12 │
│ 1323437 │ Good Harbor                                               

In [19]:
con.sql("""--sql
    SELECT 
        TRY_CAST(author_id AS INTEGER) AS author_id,
        name AS author_name,
        TRY_CAST(average_rating AS FLOAT) AS author_average_rating,
        TRY_CAST(ratings_count AS INTEGER) AS author_ratings_count,
    FROM read_json($1)
    LIMIT 5
    """,
    params=[authors_path],
).show()

┌───────────┬──────────────────┬───────────────────────┬──────────────────────┐
│ author_id │   author_name    │ author_average_rating │ author_ratings_count │
│   int32   │     varchar      │         float         │        int32         │
├───────────┼──────────────────┼───────────────────────┼──────────────────────┤
│    604031 │ Ronald J. Fields │                  3.98 │                   49 │
│    626222 │ Anita Diamant    │                  4.08 │               546796 │
│     10333 │ Barbara Hambly   │                  3.92 │               122118 │
│      9212 │ Jennifer Weiner  │                  3.68 │               888522 │
│    149918 │ Nigel Pennick    │                  3.82 │                 1740 │
└───────────┴──────────────────┴───────────────────────┴──────────────────────┘



In [20]:
con.sql("""--sql
    SELECT 
        TRY_CAST(series_id AS INTEGER) AS series_id,
        title AS series_title,
        TRY_CAST(series_works_count AS INTEGER) AS series_works_count,
        description AS series_description,
    FROM read_json($1)
    LIMIT 5
    """,
    params=[series_path],
).show()

┌───────────┬──────────────────────────────────────────┬────────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│ series_id │               series_title               │ series_works_count │                                               series_description                                                │
│   int32   │                 varchar                  │       int32        │                                                     varchar                                                     │
├───────────┼──────────────────────────────────────────┼────────────────────┼─────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│    189911 │ Sun Wolf and Starhawk                    │                  9 │                                                                                                                 │
│    151854 │ Avalon: Web of Magic      

In [ ]:
con.sql("""--sql
    SELECT 
        user_id AS user_id,
        TRY_CAST(book_id AS INTEGER) AS book_id,
        TRY_CAST(rating AS INTEGER) AS rating,
    FROM read_parquet($1)
    LIMIT 5
    """,
    params=[interactions_path],
).show()

┌──────────────────────────────────┬─────────┬────────┐
│             user_id              │ book_id │ rating │
│             varchar              │  int32  │ int32  │
├──────────────────────────────────┼─────────┼────────┤
│ 30d3b2b83ded519143a9d63d8362d26f │  408615 │      3 │
│ 30d3b2b83ded519143a9d63d8362d26f │ 2746541 │      3 │
│ 30d3b2b83ded519143a9d63d8362d26f │ 1981481 │      3 │
│ 30d3b2b83ded519143a9d63d8362d26f │  241387 │      4 │
│ 30d3b2b83ded519143a9d63d8362d26f │ 3326321 │      1 │
└──────────────────────────────────┴─────────┴────────┘



In [49]:
con.sql("""--sql
    SELECT
        TRY_CAST(book_id AS INTEGER) AS book_id,
        TRY_CAST(a.author_id AS INTEGER) AS author_id,
    FROM read_json($1)
    CROSS JOIN UNNEST(authors) AS temp(a)
    LIMIT 10
    """,
    params=[books_path],
).show()

┌──────────┬───────────┐
│ book_id  │ author_id │
│  int32   │   int32   │
├──────────┼───────────┤
│  5333265 │    604031 │
│  1333909 │    626222 │
│  7327624 │     10333 │
│  6066819 │      9212 │
│   287140 │    149918 │
│   287141 │   3041852 │
│   378460 │    215594 │
│  6066812 │     19158 │
│ 34883016 │   5807700 │
│   287149 │   2983296 │
└──────────┴───────────┘
  10 rows    2 columns



In [50]:
con.sql("""--sql
    SELECT
        TRY_CAST(book_id AS INTEGER) AS book_id,
        TRY_CAST(s AS INTEGER) AS series_id
    FROM read_json($1)
    CROSS JOIN UNNEST(series) AS temp(s)
    LIMIT 10
    """,
    params=[books_path],
).show()

┌──────────┬───────────┐
│ book_id  │ series_id │
│  int32   │   int32   │
├──────────┼───────────┤
│  7327624 │    189911 │
│  6066812 │    151854 │
│  6066814 │    169353 │
│ 33394837 │   1052227 │
│    89371 │   1070125 │
│ 12182387 │    147734 │
│ 29074697 │    953679 │
│ 29074693 │    811663 │
│  1902202 │    408775 │
│  4541271 │    250807 │
└──────────┴───────────┘
  10 rows    2 columns



In [52]:
con.sql("""--sql
    SELECT
        TRY_CAST(book_id AS INTEGER) AS book_id,
        genres.fiction AS fiction_vote
    FROM read_json($1)
    LIMIT 10
    """,
    params=[genres_path],
).show()

┌──────────┬──────────────┐
│ book_id  │ fiction_vote │
│  int32   │    int128    │
├──────────┼──────────────┤
│  5333265 │         NULL │
│  1333909 │          219 │
│  7327624 │            8 │
│  6066819 │          555 │
│   287140 │         NULL │
│   287141 │            1 │
│   378460 │            2 │
│  6066812 │            7 │
│ 34883016 │         NULL │
│   287149 │         NULL │
└──────────┴──────────────┘
  10 rows       2 columns

